# Gaussian figures for Lecture 1

Author-facing figure source for the univariate Gaussian and Book 2 §2.3 slides.
Run every cell from top to bottom. The three SVG files are saved under
`generated/` and are included by `slide_deck.typ`.

The plots are original course renderings of ideas in Murphy, *Probabilistic
Machine Learning: An Introduction* §2.6.2 and *Advanced Topics* §2.3.1.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("generated")
OUT.mkdir(exist_ok=True)
ACCENT = "#aa624d"
INK = "#302d2a"
MUTED = "#7a7771"
PAPER = "#fbfaf7"
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
    "svg.fonttype": "none",
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})


## 1. Univariate Gaussian spread

Keep the mean fixed at zero and vary the standard deviation. Each curve
integrates to one, so a narrower curve has a taller peak.

In [ ]:
x = np.linspace(-5, 5, 800)
fig, ax = plt.subplots(figsize=(8.8, 3.4), constrained_layout=True)
for sd, color in [(0.5, ACCENT), (1.0, INK), (2.0, MUTED)]:
    pdf = np.exp(-0.5 * (x / sd) ** 2) / (np.sqrt(2 * np.pi) * sd)
    ax.plot(x, pdf, lw=2.7, color=color, label=fr"$\sigma={sd:g}$")
ax.axvline(0, lw=1, color="#b6b1aa", ls="--")
ax.set(xlim=(-4.5, 4.5), ylim=(0, 0.9), xlabel="$y$", ylabel="Density")
ax.legend(frameon=False, ncol=3, loc="upper right")
ax.grid(alpha=0.15)
fig.savefig(OUT / "gaussian_spread.svg", bbox_inches="tight")
plt.close(fig)


## 2. Covariance and contour geometry

Equal Mahalanobis distance gives equal-density contours. Compare a full,
diagonal, and spherical covariance on the same coordinate scale.

In [ ]:
grid = np.linspace(-3.3, 3.3, 350)
xx, yy = np.meshgrid(grid, grid)
points = np.stack([xx, yy], axis=-1)
cases = [
    ("Full covariance", np.array([[1.4, 0.85], [0.85, 1.0]])),
    ("Diagonal covariance", np.diag([1.4, 0.55])),
    ("Spherical covariance", 0.85 * np.eye(2)),
]
fig, axes = plt.subplots(1, 3, figsize=(10.8, 3.8), constrained_layout=True)
for ax, (title, cov) in zip(axes, cases):
    q = np.einsum("...i,ij,...j->...", points, np.linalg.inv(cov), points)
    ax.contour(xx, yy, q, levels=[1, 2.5, 5], colors=[ACCENT], linewidths=[2.2, 1.8, 1.4])
    ax.plot(0, 0, "o", color=INK, ms=4)
    ax.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), aspect="equal", title=title)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_xticks([-3, 0, 3])
    ax.set_yticks([-3, 0, 3])
    ax.grid(alpha=0.12)
fig.savefig(OUT / "gaussian_contours.svg", bbox_inches="tight")
plt.close(fig)


## 3. High-dimensional Gaussian radius

For a standard Gaussian in $D$ dimensions, plot the simulated radius divided
by $\sqrt{D}$. Its relative spread narrows as dimension increases. Fixed
seed makes the plot reproducible.

In [ ]:
rng = np.random.default_rng(20261001)
fig, ax = plt.subplots(figsize=(8.8, 3.4), constrained_layout=True)
for d, color in [(2, MUTED), (20, INK), (100, ACCENT)]:
    samples = rng.standard_normal((60_000, d))
    radius_ratio = np.linalg.norm(samples, axis=1) / np.sqrt(d)
    ax.hist(radius_ratio, bins=95, range=(0, 2.3), density=True,
            histtype="step", lw=2.4, color=color, label=fr"$D={d}$")
ax.axvline(1, color="#b6b1aa", lw=1, ls="--")
ax.set(xlim=(0, 2.3), xlabel=r"Radius $R/\sqrt{D}$", ylabel="Density")
ax.legend(frameon=False, ncol=3, loc="upper right")
ax.grid(alpha=0.15)
fig.savefig(OUT / "gaussian_shell_radius.svg", bbox_inches="tight")
plt.close(fig)


## Checks

The three generated files should exist and be nonempty. The radius figure
shows concentration **relative to** $\sqrt{D}$; it does not claim the
unnormalized radial width vanishes.

In [ ]:
for name in ["gaussian_spread.svg", "gaussian_contours.svg", "gaussian_shell_radius.svg"]:
    file = OUT / name
    assert file.exists() and file.stat().st_size > 1000, file
    print(file, file.stat().st_size, "bytes")
